# Quick Start — UrbanSolarCarver

UrbanSolarCarver (USC) generates **maximum building volumes** that respect solar access, daylight, and thermal comfort constraints in urban environments.

You give it two meshes:
- A **maximum volume** — the largest shape you're allowed to build (e.g. a zoning envelope)
- **Test surfaces** — neighbouring facades, courtyards, or public spaces that need sunlight

USC then **carves away** the parts of the volume that block too much sun or daylight from reaching those surfaces. What remains is the **carved volume**: the largest buildable mass that satisfies your performance criteria (e.g. solar access to test surfaces).

### Available modes

USC supports 6 carving modes. Each defines *what kind* of solar/daylight access you're protecting:

| Mode | What it protects | Needs weather file? |
|------|-----------------|:-------------------:|
| `daylight` | Diffuse sky access (CIE overcast sky model) | No |
| `tilted_plane` | Geometric daylight plane (height-to-width ratio) | No |
| `time-based` | Direct sun during specific hours (Knowles solar envelope) | Yes |
| `irradiance` | Total solar radiation in Wh/m² | Yes |
| `benefit` | Heating-season solar gain minus summer penalty | Yes |
| `radiative_cooling` | Night-sky cooling potential *(experimental)* | Yes |

This notebook runs the simplest possible example to get you started.


## 1. Imports

Three libraries are needed. Two are general Python tools, one is USC itself.


In [ ]:
# pathlib — Python's built-in way to handle file paths across operating systems.
# We use it so the same code works on Windows, Mac, and Linux.
from pathlib import Path

# USC — the carving library.
# user_config: a configuration object — you set the parameters, it validates them
# load_config: Alternatively you can import load_config. It reads a YAML configuration file (pre-defined user inputs) and validates it. You can find sample config files in the `configs/` directory.
# run_pipeline: runs all three pipeline stages (preprocessing → thresholding → exporting) in one call.
# if you want to run each stage seperately to control thresholding and exporting then you must import the individual functions from each stage. See the API documentation for details.
from urbansolarcarver import user_config, run_pipeline

# trimesh — a 3D mesh library. We use it to load PLY files and view them in 3D.
import trimesh

## 2. Set up paths

The example meshes and weather data live inside the repository under `examples/`.
We point `REPO_ROOT` at the repo root so we can build paths relative to it.

 **On your own projects** you'd just use your own absolute paths — this `REPO_ROOT` pattern is only a convenience for the example notebooks.

In [ ]:

# This notebook lives in examples/, so one level up is the repo root
REPO_ROOT = Path.cwd().parent
print("Repository root:", REPO_ROOT)

# --- Input meshes ---

# max_volume: the buildable envelope — what we START with before any carving.
# Think of it as the largest possible bounding box around your object of interest (a building, a plot, an urban block etc).
max_volume_path = REPO_ROOT / "examples" / "test_inputs" / "full_blocks" / "maxVolume.ply"

# test_surfaces: surfaces that are evaluated for some performance metric (depending on mode selected).
# USC will carve the max volume to protect environmental performance of these surfaces.
test_surface_path = REPO_ROOT / "examples" / "test_inputs" / "full_blocks" / "testSurfaces.ply"

# --- Weather data ---

# EPW (EnergyPlus Weather) files contain hourly climate data for a location.
# Modes that depend on sun position or temperature need one.
# Download from ladybug.tools/epwmap/
epw_path = str(ADDRESS_TO_EPW_FILE)

print(f"Max volume:    {max_volume_path}")
print(f"Test surfaces: {test_surface_path}")
print(f"EPW file:      {epw_path}")


## 3. Preview the input geometry

Before running anything, let's see what we're working with.


In [ ]:
# Load the two PLY meshes
max_volume = trimesh.load(max_volume_path)
test_surfaces = trimesh.load(test_surface_path)

# Color them so we can tell them apart in the viewer
max_volume.visual.face_colors = [255, 100, 100, 180]     # semi-transparent red
test_surfaces.visual.face_colors = [100, 255, 100, 255]  # solid green

print(f"Max volume:    {len(max_volume.vertices)} vertices, {len(max_volume.faces)} faces")
print(f"Test surfaces: {len(test_surfaces.vertices)} vertices, {len(test_surfaces.faces)} faces")

# Show them together — red = what we'll carve, green = what needs sun
scene = trimesh.Scene([max_volume, test_surfaces])
scene.show()

#NOTE: surfaces only visible when normal vectors point toward the camera, try rotating the view.


## 4. Configure the run

USC is configured through a `user_config` object. You set the parameters you care about — everything else has sensible defaults.

### What we're setting and why

| Setting | Value | Why |
|---------|-------|-----|
| `mode` | `"benefit"` | Weighs solar gain by heating demand — the most common mode for urban massing |
| `voxel_size` | 1.0 m | Each voxel is a 1 m cube. Smaller = finer but slower. Start with 2.0 for quick tests. |
| `grid_step` | 1.0 m | How densely the test surfaces are sampled. Must be ≤ `voxel_size`. |
| `balance_temperature` | 20.0 °C | Below this outdoor temperature, the building needs heating. Solar gain during those hours counts as beneficial. |
| `balance_offset` | 2.0 °C | Dead-band: only hours where `T_air < 18 °C` (= 20 − 2) are credited. Avoids counting mild days. |
| `carve_fraction` | 0.7 | Remove voxels responsible for 70% of total solar obstruction |
| Analysis period | Oct 15 – Apr 15, 7:00–19:00 | The heating season — all daylight hours during months when heating is typically needed in a temperate climate |


In [ ]:
# Where to save the output
out_dir = REPO_ROOT / "examples" / "test_outputs" / "quick_start"

# Build the configuration directly — no YAML file needed
cfg = user_config(
    # The two input meshes + weather file
    max_volume_path   = str(max_volume_path),
    test_surface_path = str(test_surface_path),
    epw_path          = epw_path,
    out_dir           = str(out_dir),

    # Carving mode
    mode = "benefit",

    # Heating season (when passive solar gain matters)
    start_month = 10, start_day = 15, start_hour = 7,
    end_month   = 4,  end_day   = 15, end_hour   = 19,

    # Benefit-specific: what counts as "cold enough to need heating"
    balance_temperature = 20.0,  # °C — building balance point
    balance_offset      = 2.0,   # °C — dead-band below balance point

    # Resolution
    voxel_size = 1.0,  # metres per voxel edge
    grid_step  = 1.0,  # test-surface sampling spacing (must be ≤ voxel_size)

    # Diagnostics
    diagnostics = True,        # write diagnostic JSON
    diagnostic_plots = True,   # generate score histograms and sky patch plots
)

#Print config. overrides
print(f"Mode:               {cfg.mode}")
print(f"Voxel size:         {cfg.voxel_size} m")
print(f"Grid step:          {cfg.grid_step} m")
print(f"Balance temp:       {cfg.balance_temperature} °C (offset {cfg.balance_offset} °C)")
print(f"Effective cutoff:   T_air < {cfg.balance_temperature - cfg.balance_offset} °C")
print(f"Carve fraction:     {cfg.carve_fraction}")
print(f"Analysis period:    {cfg.start_day}/{cfg.start_month} → {cfg.end_day}/{cfg.end_month}")


## 5. Run the pipeline

`run_pipeline()` executes three stages in sequence:

| Stage | What it does | Speed |
|-------|-------------|-------|
| **Preprocessing** | Voxelizes the volume, samples the test surfaces, traces millions of rays to compute a **score** per voxel (how much solar access it blocks) | Relatively fast (GPU) |
| **Thresholding** | Converts scores to a binary keep/carve decision per voxel | Instant |
| **Exporting** | Reconstructs a triangle mesh from the surviving voxels, saves to disk | Fast |

This separation matters: if you later want to try a different `carve_fraction`, you only re-run thresholding + exporting (see Tutorial 4).


In [ ]:
# Run all three stages in one call
result = run_pipeline(cfg, cfg.out_dir)

print(f"Mesh saved to:   {result.export_path}")
print(f"Volume retained: {result.retention_pct:.1f}%")
print(f"Faces:           {result.faces:,}")


## 6. View the result

The carved mesh is the **benefit envelope** — the maximum buildable volume that preserves beneficial solar access on the test surfaces during the heating season.

In [ ]:
carved = trimesh.load(result.export_path)
carved.visual.face_colors = [70, 130, 180, 220]  # steel blue

# show carved mesh
print(f"Carved mesh: {len(carved.vertices)} vertices, {len(carved.faces)} faces")
trimesh.Scene([carved]).show()

## 7. Run report

Every pipeline run writes a `run_report.md` — a human-readable summary of configuration, scores, thresholding decisions, mesh statistics, and timings.

In [ ]:
from IPython.display import Markdown, display

report_path = Path(cfg.out_dir) / "run_report.md" # the report is saved in the output directory
if report_path.is_file():
    display(Markdown(report_path.read_text(encoding="utf-8")))
else:
    print(f"Report not found at {report_path}")


## 8. Diagnostic plots

When `diagnostic_plots=True`, USC generates visualizations of the score distribution and sky patch weights.
These help verify that the analysis period, mode, and threshold are working as expected.

In [ ]:
from IPython.display import Image, display
import json as _json

# Each pipeline stage writes a diagnostic.json that records what it did
# and where it saved any plots. We read those to find the image paths.

diag_dir = Path(cfg.out_dir)

# Thresholding diagnostic: score distribution with the threshold line marked.
# Voxels to the right of the line were carved away.
thresholding_diag = _json.loads((diag_dir / "thresholding" / "diagnostics" / "diagnostic.json").read_text())
display(Image(filename=thresholding_diag["threshold_histogram"]))

# Preprocessing diagnostic: sky patch weight maps.
# Shows which parts of the sky dome contributed most to the obstruction scores.
preprocessing_diag = _json.loads((diag_dir / "preprocessing" / "diagnostics" / "diagnostic.json").read_text())
for img in preprocessing_diag["sky_patch_images"]:
    display(Image(filename=img))

## What just happened?

USC scored every voxel in the building volume by how much beneficial solar radiation it blocks from reaching the test surfaces during the heating season. It then carved away the worst offenders: the voxels responsible for 70% of the total obstruction score (`carve_fraction = 0.7`).

Because the score distribution is usually right-skewed (a few voxels cause most of the blockage), removing 70% of the *score* typically removes far fewer than 70% of the *voxels*. That's why the retained volume is usually well above 30%.

## Next steps

- **Solar envelope**: the original time-based concept → Tutorial 2
- **Daylight envelopes**: geometric + CIE overcast → Tutorial 3
- **Chaining modes**: combine solar + daylight constraints → Tutorial 4
- **Advanced**: preprocess once, re-threshold many times → Tutorial 4
- **CLI**: `usc run -c config.yaml` does the same thing from the terminal
- **Grasshopper**: see `grasshopper/` for the Rhino/GH integration
